# PromptCraft-SeqRec — Beat the Baseline on Kaggle T4 GPU

**Goal**: Show that richer item description prompts for BGE-M3 embeddings beat the paper's title-only baseline on NDCG@10, HR@10, MRR.

**Pipeline**:
1. Download Amazon Beauty data (reviews + metadata)
2. Build SessionDataset (replicate the paper's preprocessing)
3. Generate BGE-M3 embeddings for 6 prompt strategies
4. Apply PCA (256-dim) per strategy
5. Train LLM2SASRec for each strategy
6. Compare NDCG@10, HR@10, MRR across strategies

In [ ]:
# ─── Cell 1: Environment Setup ───────────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO_URL = "https://github.com/faroukq1/LLM-Sequential-Recommendation.git"
REPO_DIR = Path("/kaggle/working/LLM-Sequential-Recommendation")

if not REPO_DIR.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repo already exists.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Install BGE-M3 (FlagEmbedding) — uses GPU automatically
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "FlagEmbedding>=1.2.0", "sentence-transformers", "scikit-learn"],
    check=True
)

print("Setup complete. Working dir:", os.getcwd())

In [ ]:
# ─── Cell 2: Imports & Config ─────────────────────────────────────────────────
import ast, gc, gzip, json, pickle, re, time, urllib.request, warnings
from collections import defaultdict
from datetime import datetime
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────────
WORK_DIR   = Path("/kaggle/working/promptcraft_beauty")
DATA_DIR   = WORK_DIR / "data"
EMB_DIR    = WORK_DIR / "embeddings"
RESULTS_DIR = WORK_DIR / "results"
FIGURES_DIR = WORK_DIR / "figures"
for d in [DATA_DIR, EMB_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Data URLs (2014 Amazon Beauty dataset, same as paper) ─────────────────────
REVIEWS_URL = (
    "http://snap.stanford.edu/data/amazon/productGraph/"
    "categoryFiles/reviews_Beauty_5.json.gz"
)
META_URL = (
    "http://snap.stanford.edu/data/amazon/productGraph/"
    "categoryFiles/meta_Beauty.json.gz"
)

# ── Experiment Config ─────────────────────────────────────────────────────────
MIN_INTERACTIONS   = 5       # k-core filtering threshold
TOP_KS             = [10, 20]
EMB_DIM            = 256     # PCA target dimension
MAX_SEQ_LEN        = 20
NUM_EPOCHS         = 15
FIT_BATCH_SIZE     = 256
PRED_BATCH_SIZE    = 2048
TRAIN_VAL_FRACTION = 0.1
EARLY_STOPPING     = 3
LEARNING_RATE      = 3e-4
WEIGHT_DECAY       = 0.05
DROP_RATE          = 0.0
CORES              = 2
VERBOSE            = True
TEST_FRAC          = 0.2

print("Config ready.")
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
# ─── Cell 3: Local Model Implementations ─────────────────────────────────────
# Self-contained — no import from repo modules needed for models.
# Pattern adapted from kaggle/core_models_ml100k_single_notebook.ipynb

def _coerce_embedding(value: Any) -> np.ndarray:
    if isinstance(value, np.ndarray):
        return value.astype(np.float32)
    if isinstance(value, list):
        return np.asarray(value, dtype=np.float32)
    if isinstance(value, str):
        return np.asarray(json.loads(value), dtype=np.float32)
    raise TypeError(f"Unsupported embedding type: {type(value)}")


def _pad_left(values: list[int], length: int, pad_value: int = 0) -> np.ndarray:
    if len(values) >= length:
        return np.asarray(values[-length:], dtype=np.int32)
    return np.asarray(([pad_value] * (length - len(values))) + values, dtype=np.int32)


def _top_k_indices(scores: np.ndarray, top_k: int) -> np.ndarray:
    top_k = min(top_k, len(scores))
    idx = np.argpartition(scores, -top_k)[-top_k:]
    return idx[np.argsort(scores[idx])[::-1]]


class NeuralSequenceRecommender:
    def __init__(
        self,
        model_name: str,
        architecture: str,
        N: int = 20,
        L: int = 1,
        h: int = 2,
        emb_dim: int = 256,
        hidden_dim: int = 256,
        drop_rate: float = 0.0,
        mask_prob: float = 0.4,
        num_epochs: int = 15,
        fit_batch_size: int = 256,
        pred_batch_size: int = 2048,
        train_val_fraction: float = 0.1,
        early_stopping_patience: int = 3,
        learning_rate: float = 3e-4,
        weight_decay: float = 0.05,
        optimizer_kwargs: dict | None = None,
        activation: str = "relu",
        pred_seen: bool = False,
        is_verbose: bool = True,
        cores: int = 2,
        transformer_layer_kwargs: dict | None = None,
        **_: dict,
    ) -> None:
        self.model_name = model_name
        self.architecture = architecture
        self.N = int(N)
        self.L = int(L)
        self.h = int(h)
        self.emb_dim = int(emb_dim)
        self.hidden_dim = int(hidden_dim)
        self.drop_rate = float(drop_rate)
        self.mask_prob = float(mask_prob)
        self.num_epochs = int(num_epochs)
        self.fit_batch_size = int(fit_batch_size)
        self.pred_batch_size = int(pred_batch_size)
        self.train_val_fraction = float(train_val_fraction)
        self.early_stopping_patience = int(early_stopping_patience)
        self.learning_rate = float(learning_rate)
        self.weight_decay = float(weight_decay)
        self.optimizer_kwargs = optimizer_kwargs or {}
        self.activation = activation
        self.pred_seen = pred_seen
        self.is_verbose = is_verbose
        self.transformer_layer_kwargs = transformer_layer_kwargs or {}
        self.model: keras.Model | None = None
        self.item_to_idx: dict[int, int] = {}
        self.idx_to_item: dict[int, int] = {}
        self.popular_items: list[int] = []
        self.is_trained = False

    def _build_initial_embedding_matrix(self) -> np.ndarray | None:
        return None

    def _build_training_examples(self, train_df: pd.DataFrame):
        x_rows, y_rows = [], []
        sort_cols = ["SessionId", "Time", "ItemId"] if "Time" in train_df.columns else ["SessionId", "ItemId"]
        ordered = train_df.sort_values(sort_cols).reset_index(drop=True)
        for _, session_df in ordered.groupby("SessionId", sort=False):
            idx_items = [self.item_to_idx[int(i)] for i in session_df["ItemId"] if int(i) in self.item_to_idx]
            if len(idx_items) < 2:
                continue
            for t in range(1, len(idx_items)):
                seq = idx_items[max(0, t - self.N):t]
                x_rows.append(_pad_left(seq, self.N))
                y_rows.append(idx_items[t])
        if not x_rows:
            raise ValueError("No training sequences found.")
        return np.vstack(x_rows).astype(np.int32), np.asarray(y_rows, dtype=np.int32)

    def _build_model(self, num_items_plus_pad: int, initial_embeddings: np.ndarray | None = None) -> keras.Model:
        inputs = keras.Input(shape=(self.N,), dtype="int32")
        emb_layer = keras.layers.Embedding(
            input_dim=num_items_plus_pad,
            output_dim=self.emb_dim,
            mask_zero=True,
            name="item_emb",
        )
        x = emb_layer(inputs)
        pos_indices = tf.range(start=0, limit=self.N, delta=1)
        pos_emb = keras.layers.Embedding(self.N, self.emb_dim, name="pos_emb")(pos_indices)
        x = x + pos_emb
        for _ in range(max(1, self.L)):
            attn = keras.layers.MultiHeadAttention(
                num_heads=max(1, self.h),
                key_dim=max(4, self.emb_dim // max(1, self.h)),
                dropout=self.drop_rate,
            )(x, x)
            x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn)
            ff = keras.layers.Dense(self.hidden_dim, activation=self.activation)(x)
            ff = keras.layers.Dropout(self.drop_rate)(ff)
            ff = keras.layers.Dense(self.emb_dim)(ff)
            x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ff)
        x = keras.layers.Lambda(lambda t: t[:, -1, :])(x)
        outputs = keras.layers.Dense(num_items_plus_pad, activation="softmax")(x)
        model = keras.Model(inputs=inputs, outputs=outputs)
        if initial_embeddings is not None:
            emb_layer.build((None, self.N))
            emb_layer.set_weights([initial_embeddings.astype(np.float32)])
        lr = float(self.optimizer_kwargs.get("learning_rate", self.learning_rate))
        wd = float(self.optimizer_kwargs.get("weight_decay", self.weight_decay))
        if wd > 0 and hasattr(keras.optimizers, "AdamW"):
            optimizer = keras.optimizers.AdamW(learning_rate=lr, weight_decay=wd)
        else:
            optimizer = keras.optimizers.Adam(learning_rate=lr)
        model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy")
        return model

    def train(self, train_data: pd.DataFrame, item_data: pd.DataFrame | None = None) -> None:
        tf.keras.utils.set_random_seed(42)
        unique_items = sorted(train_data["ItemId"].astype(int).unique().tolist())
        self.item_to_idx = {item: idx + 1 for idx, item in enumerate(unique_items)}
        self.idx_to_item = {idx: item for item, idx in self.item_to_idx.items()}
        self.popular_items = train_data["ItemId"].value_counts().index.astype(int).tolist()
        x_train, y_train = self._build_training_examples(train_data)
        initial_embeddings = self._build_initial_embedding_matrix()
        self.model = self._build_model(
            num_items_plus_pad=len(self.item_to_idx) + 1,
            initial_embeddings=initial_embeddings,
        )
        callbacks = []
        if self.early_stopping_patience > 0:
            callbacks.append(keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=self.early_stopping_patience,
                restore_best_weights=True,
            ))
        val_split = self.train_val_fraction if self.train_val_fraction > 0 else 0.0
        self.model.fit(
            x_train, y_train,
            epochs=self.num_epochs,
            batch_size=min(self.fit_batch_size, len(x_train)),
            validation_split=val_split,
            verbose=1 if self.is_verbose else 0,
            callbacks=callbacks,
        )
        self.is_trained = True

    def predict(self, predict_data: dict[int, np.ndarray], top_k: int = 10) -> dict[int, np.ndarray]:
        if not self.is_trained:
            raise RuntimeError(f"{self.model_name} not trained.")
        case_ids = list(predict_data.keys())
        x_input = np.vstack([
            _pad_left(
                [self.item_to_idx[int(i)] for i in predict_data[cid] if int(i) in self.item_to_idx],
                self.N,
            )
            for cid in case_ids
        ]).astype(np.int32)
        probs = self.model.predict(x_input, batch_size=min(self.pred_batch_size, len(case_ids)), verbose=0)
        predictions: dict[int, np.ndarray] = {}
        for row_idx, case_id in enumerate(case_ids):
            scores = probs[row_idx].copy()
            scores[0] = -np.inf  # mask pad
            if not self.pred_seen:
                for item in predict_data[case_id]:
                    idx = self.item_to_idx.get(int(item))
                    if idx is not None:
                        scores[idx] = -np.inf
            top_idx = _top_k_indices(scores, top_k)
            recs = [self.idx_to_item[int(idx)] for idx in top_idx if int(idx) in self.idx_to_item]
            if len(recs) < top_k:
                seen = set(int(i) for i in predict_data[case_id])
                for item in self.popular_items:
                    if item not in seen and item not in recs:
                        recs.append(item)
                    if len(recs) == top_k:
                        break
            predictions[int(case_id)] = np.asarray(recs[:top_k], dtype=int)
        return predictions

    def name(self) -> str:
        return self.model_name


class SASRec(NeuralSequenceRecommender):
    def __init__(self, **kwargs):
        super().__init__(model_name="SASRec", architecture="sas", **kwargs)


class LLM2SASRec(NeuralSequenceRecommender):
    """SASRec initialized with pre-computed (PCA-reduced) BGE-M3 embeddings."""
    def __init__(self, embedding_csv_path: str, **kwargs):
        self.external_embeddings = self._load_embeddings(embedding_csv_path)
        super().__init__(model_name="LLM2SASRec", architecture="sas", **kwargs)

    @staticmethod
    def _load_embeddings(path: str) -> dict[int, np.ndarray]:
        df = pd.read_csv(path)
        return {int(row["ItemId"]): _coerce_embedding(row["embedding"]) for _, row in df.iterrows()}

    def _build_initial_embedding_matrix(self) -> np.ndarray:
        rng = np.random.default_rng(42)
        matrix = rng.normal(0.0, 0.01, size=(len(self.item_to_idx) + 1, self.emb_dim)).astype(np.float32)
        matrix[0] = 0.0
        for item, idx in self.item_to_idx.items():
            vec = self.external_embeddings.get(int(item))
            if vec is None:
                continue
            if vec.shape[0] < self.emb_dim:
                vec = np.pad(vec, (0, self.emb_dim - vec.shape[0]))
            elif vec.shape[0] > self.emb_dim:
                vec = vec[:self.emb_dim]
            matrix[idx] = vec.astype(np.float32)
        return matrix


print("Local model classes loaded: SASRec, LLM2SASRec")

In [ ]:
# ─── Cell 4: Evaluation Helpers ───────────────────────────────────────────────
# Standalone NDCG@K, HR@K, MRR — no repo imports needed.

def ndcg_at_k(recommendations: dict[int, np.ndarray], ground_truths: dict[int, np.ndarray], k: int) -> float:
    scores = []
    for sid, gt in ground_truths.items():
        recs = recommendations.get(sid, np.array([]))
        gt_set = set(int(x) for x in gt)
        dcg = 0.0
        for rank, item in enumerate(recs[:k]):
            if int(item) in gt_set:
                dcg += 1.0 / np.log2(rank + 2)
        ideal = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt_set), k)))
        scores.append(dcg / ideal if ideal > 0 else 0.0)
    return float(np.mean(scores)) if scores else 0.0


def hr_at_k(recommendations: dict[int, np.ndarray], ground_truths: dict[int, np.ndarray], k: int) -> float:
    scores = []
    for sid, gt in ground_truths.items():
        recs = recommendations.get(sid, np.array([]))
        gt_set = set(int(x) for x in gt)
        scores.append(1.0 if any(int(r) in gt_set for r in recs[:k]) else 0.0)
    return float(np.mean(scores)) if scores else 0.0


def mrr(recommendations: dict[int, np.ndarray], ground_truths: dict[int, np.ndarray], k: int = 20) -> float:
    scores = []
    for sid, gt in ground_truths.items():
        recs = recommendations.get(sid, np.array([]))
        gt_set = set(int(x) for x in gt)
        rr = 0.0
        for rank, item in enumerate(recs[:k]):
            if int(item) in gt_set:
                rr = 1.0 / (rank + 1)
                break
        scores.append(rr)
    return float(np.mean(scores)) if scores else 0.0


def evaluate(recommendations, ground_truths, top_ks=(10, 20)):
    results = {}
    for k in top_ks:
        results[f"NDCG@{k}"] = ndcg_at_k(recommendations, ground_truths, k)
        results[f"HR@{k}"]   = hr_at_k(recommendations, ground_truths, k)
    results["MRR"] = mrr(recommendations, ground_truths)
    return results


print("Evaluation functions loaded: ndcg_at_k, hr_at_k, mrr, evaluate")

In [ ]:
# ─── Cell 5: Download Amazon Beauty Data ─────────────────────────────────────
import urllib.request

reviews_gz_path = DATA_DIR / "reviews_Beauty_5.json.gz"
meta_gz_path    = DATA_DIR / "meta_Beauty.json.gz"

def download_if_missing(url: str, dest: Path) -> None:
    if dest.exists():
        print(f"[SKIP] {dest.name} already exists.")
        return
    print(f"Downloading {dest.name} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"  -> Saved to {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

download_if_missing(REVIEWS_URL, reviews_gz_path)
download_if_missing(META_URL, meta_gz_path)

In [ ]:
# ─── Cell 6: Parse Reviews & Build Sessions ───────────────────────────────────
# Replicate beauty/create_sessions.ipynb logic exactly.

def load_reviews_gz(path: Path) -> pd.DataFrame:
    rows = []
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for line in f:
            try:
                rec = json.loads(line.strip())
                rows.append({
                    "SessionId": rec["reviewerID"],
                    "ItemId": rec["asin"],
                    "Time": rec["unixReviewTime"],
                    "Reward": 1.0,
                })
            except:
                pass
    df = pd.DataFrame(rows)
    df.sort_values("Time", inplace=True)
    print(f"Loaded {len(df):,} interactions, {df['SessionId'].nunique():,} users, {df['ItemId'].nunique():,} items")
    return df


def kcore_filter(df: pd.DataFrame, k: int) -> pd.DataFrame:
    prev = len(df)
    while True:
        user_counts = df.groupby("SessionId")["ItemId"].count()
        item_counts = df.groupby("ItemId")["SessionId"].count()
        valid_users = user_counts[user_counts >= k].index
        valid_items = item_counts[item_counts >= k].index
        df = df[df["SessionId"].isin(valid_users) & df["ItemId"].isin(valid_items)]
        if len(df) == prev:
            break
        prev = len(df)
    print(f"After {k}-core: {len(df):,} interactions, {df['SessionId'].nunique():,} users, {df['ItemId'].nunique():,} items")
    return df.copy()


# Load and filter
raw_df = load_reviews_gz(reviews_gz_path)
filtered_df = kcore_filter(raw_df, MIN_INTERACTIONS)

# Map string IDs to integers
user_ids = sorted(filtered_df["SessionId"].unique())
item_ids = sorted(filtered_df["ItemId"].unique())
user_to_int = {u: i for i, u in enumerate(user_ids)}
item_to_int = {it: i for i, it in enumerate(item_ids)}
int_to_item = {i: it for it, i in item_to_int.items()}  # int → ASIN

filtered_df["SessionId"] = filtered_df["SessionId"].map(user_to_int)
filtered_df["ItemId"] = filtered_df["ItemId"].map(item_to_int)

# Convert Time to proper datetime string
filtered_df["Time"] = pd.to_datetime(filtered_df["Time"], unit='s').dt.strftime("%Y-%m-%d %H:%M:%S.%f")

sessions_csv = DATA_DIR / "sessions.csv"
filtered_df[["SessionId", "ItemId", "Time", "Reward"]].to_csv(sessions_csv, index=False)
print(f"Sessions saved to {sessions_csv}")
print(f"Total: {len(user_ids):,} users, {len(item_ids):,} items")

In [ ]:
# ─── Cell 7: Build SessionDataset & Train/Test Split ─────────────────────────
from main.data.session_dataset import SessionDataset
from main.data.temporal_split import TemporalSplit

dataset = SessionDataset(str(sessions_csv), n_withheld=1)
dataset.load_and_split(TemporalSplit(test_frac=TEST_FRAC, num_folds=0, filter_non_trained_test_items=True))

train_data       = dataset.get_train_data()
test_prompts     = dataset.get_test_prompts()
test_gts         = dataset.get_test_ground_truths()
num_unique_items = dataset.get_unique_item_count()

print(f"Train interactions: {len(train_data):,}")
print(f"Test cases: {len(test_prompts):,}")
print(f"Unique items: {num_unique_items:,}")

In [ ]:
# ─── Cell 8: Parse Amazon Metadata ───────────────────────────────────────────
# meta_Beauty.json uses Python dict literals (single quotes) — need robust parser.

def parse_meta_line(line: str) -> dict | None:
    line = line.strip()
    if not line:
        return None
    # Try standard JSON first (some entries are valid JSON)
    try:
        return json.loads(line)
    except:
        pass
    # Try ast.literal_eval (handles Python dict format)
    try:
        return ast.literal_eval(line)
    except:
        pass
    # Last resort: aggressive cleanup
    try:
        fixed = re.sub(r"(?<![\\])'", '"', line)
        fixed = fixed.replace("True", "true").replace("False", "false").replace("None", "null")
        return json.loads(fixed)
    except:
        return None


def load_metadata_gz(path: Path, valid_asins: set) -> dict:
    """Load metadata for items in valid_asins. Returns {asin: metadata_dict}."""
    meta = {}
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            rec = parse_meta_line(line)
            if rec is None:
                continue
            asin = rec.get("asin")
            if asin and asin in valid_asins:
                meta[asin] = rec
    print(f"Loaded metadata for {len(meta):,} / {len(valid_asins):,} items")
    return meta


valid_asins = set(int_to_item.values())  # ASINs (strings) of all items in dataset
item_meta = load_metadata_gz(meta_gz_path, valid_asins)

# Show sample
sample_asin = next(iter(item_meta))
print(f"\nSample item [{sample_asin}]:")
sample_fields = {k: str(v)[:100] for k, v in item_meta[sample_asin].items()}
for k, v in sample_fields.items():
    print(f"  {k}: {v}")

In [ ]:
# ─── Cell 9: Prompt Strategies ───────────────────────────────────────────────
# item_int_id → prompt text
# We use int_to_item to get ASIN, then look up metadata.

def _safe_get(d: dict, key: str, default: str = "unknown") -> str:
    val = d.get(key, default)
    if not val or val == "" or val == []:
        return default
    if isinstance(val, list):
        val = val[0] if val else default
    return str(val).strip()


def _get_categories(d: dict, n: int = 3) -> str:
    cats = d.get("categories", d.get("category", []))
    if not cats:
        return "general"
    if isinstance(cats[0], list):
        flat = [c for sub in cats for c in sub]
    else:
        flat = cats
    # drop first generic category (usually "Beauty" or top-level)
    meaningful = [c for c in flat if len(c) > 2][1:n+1]
    return ", ".join(meaningful) if meaningful else ", ".join(str(c) for c in flat[:n])


def _get_description(d: dict, max_len: int = 200) -> str:
    desc = d.get("description", "")
    if isinstance(desc, list):
        desc = " ".join(desc)
    return str(desc)[:max_len].strip()


def _get_related_titles(d: dict, meta_lookup: dict, n: int = 3) -> str:
    related = d.get("related", {})
    also_bought = related.get("also_bought", related.get("bought_together", []))
    titles = []
    for asin in also_bought[:n]:
        m = meta_lookup.get(asin)
        if m:
            t = _safe_get(m, "title")
            if t != "unknown":
                titles.append(t)
    return ", ".join(titles) if titles else "comparable products in this category"


STRATEGY_NAMES = [
    "type1_title_only",
    "type2_structured",
    "type3_rich_prose",
    "type4_user_centric",
    "type5_comparative",
    "type6_hybrid",
    "type7_structured_comparative",
]

STRATEGY_LABELS = {
    "type1_title_only":  "T1: Title Only (Baseline)",
    "type2_structured":  "T2: Structured Attrs",
    "type3_rich_prose":  "T3: Rich Prose",
    "type4_user_centric": "T4: User Centric",
    "type5_comparative": "T5: Comparative",
    "type6_hybrid":      "T6: Hybrid (Best)",
    "type7_structured_comparative": "T7: Struct+Compare",
}


def build_prompt(strategy: str, item_int: int, meta_lookup: dict) -> str:
    asin = int_to_item.get(item_int, "")
    d = meta_lookup.get(asin, {})
    title = _safe_get(d, "title", asin or f"item_{item_int}")
    brand = _safe_get(d, "brand")
    cats  = _get_categories(d)
    desc  = _get_description(d)
    price = _safe_get(d, "price")

    if strategy == "type1_title_only":
        return title

    elif strategy == "type2_structured":
        parts = [title]
        if brand != "unknown":
            parts.append(f"Brand: {brand}")
        if cats != "general":
            parts.append(f"Category: {cats}")
        if price != "unknown":
            parts.append(f"Price: {price}")
        return " | ".join(parts)

    elif strategy == "type3_rich_prose":
        base = f"{title}"
        if brand != "unknown":
            base += f" by {brand}"
        if cats != "general":
            base += f", a {cats} product"
        if desc:
            base += f". {desc}"
        return base.strip()

    elif strategy == "type4_user_centric":
        text = f"Users who like {title} enjoy"
        if cats != "general":
            text += f": {cats}"
        if brand != "unknown":
            text += f" products from {brand}"
        return text

    elif strategy == "type5_comparative":
        similar = _get_related_titles(d, meta_lookup, n=3)
        text = f"{title} is similar to: {similar}"
        if cats != "general":
            text += f". Appeals to fans of: {cats}"
        return text

    elif strategy == "type6_hybrid":
        parts = [title]
        if cats != "general":
            parts.append(f"Category: {cats}")
        if brand != "unknown":
            parts.append(f"Brand: {brand}")
        return " | ".join(parts)

    elif strategy == "type7_structured_comparative":
        parts = [title]
        if cats != "general":
            parts.append(f"Category: {cats}")
        if brand != "unknown":
            parts.append(f"Brand: {brand}")
        similar = _get_related_titles(d, meta_lookup, n=2)
        parts.append(f"Similar to: {similar}")
        return " | ".join(parts)

    return title  # fallback


# Quick sanity check on all strategies for one item
sample_int = list(item_to_int.values())[0]
print(f"Strategy preview for item {sample_int} (ASIN={int_to_item.get(sample_int)}):")
for sname in STRATEGY_NAMES:
    print(f"  [{sname}] {build_prompt(sname, sample_int, item_meta)}")

In [ ]:
# ─── Cell 10: Generate BGE-M3 Embeddings ─────────────────────────────────────
# Uses GPU automatically. Falls back to sentence-transformers if FlagEmbedding unavailable.

try:
    from FlagEmbedding import BGEM3FlagModel
    USE_BGE = True
    print("Using FlagEmbedding BGE-M3")
except ImportError:
    from sentence_transformers import SentenceTransformer
    USE_BGE = False
    print("Using sentence-transformers BGE-M3")


def load_embedding_model():
    if USE_BGE:
        return BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
    else:
        return SentenceTransformer("BAAI/bge-m3")


def encode_texts(model, texts: list[str], batch_size: int = 256) -> np.ndarray:
    if USE_BGE:
        out = model.encode(
            texts,
            batch_size=batch_size,
            max_length=128,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
        return out["dense_vecs"]
    else:
        return model.encode(texts, batch_size=batch_size, show_progress_bar=True)


# All item integer IDs in the dataset
all_item_ints = sorted(item_to_int.values())
print(f"Items to embed: {len(all_item_ints):,}")

# Load model once
bge_model = load_embedding_model()
print("BGE-M3 model loaded.")

# Generate raw embeddings for each strategy
raw_embeddings = {}  # strategy_name -> np.ndarray [n_items, 1024]

for strategy in STRATEGY_NAMES:
    raw_path = EMB_DIR / f"beauty_{strategy}_raw.npy"
    if raw_path.exists():
        print(f"[SKIP] {strategy} raw embeddings already exist.")
        raw_embeddings[strategy] = np.load(raw_path)
        continue

    print(f"\n[ENCODING] {strategy}...")
    texts = [build_prompt(strategy, item_int, item_meta) for item_int in all_item_ints]
    print(f"  Sample[0]: {texts[0]}")
    print(f"  Sample[1]: {texts[1]}")

    t0 = time.time()
    emb = encode_texts(bge_model, texts, batch_size=512)
    dt = time.time() - t0

    np.save(raw_path, emb)
    raw_embeddings[strategy] = emb
    print(f"  Shape: {emb.shape} | Time: {dt:.1f}s | Saved: {raw_path}")

# Free GPU memory
del bge_model
gc.collect()
print("\nAll raw embeddings ready.")

In [ ]:
# ─── Cell 11: Embedding Quality Analysis ─────────────────────────────────────
# Isotropy + avg pairwise distance for each strategy (before PCA).

def embedding_isotropy(emb: np.ndarray) -> float:
    """min(S) / max(S) of centered SVD. Higher = more isotropic."""
    centered = emb - emb.mean(axis=0)
    try:
        _, S, _ = np.linalg.svd(centered, full_matrices=False)
        return float(S[-1] / S[0]) if S[0] > 0 else 0.0
    except:
        return 0.0


def avg_pairwise_distance(emb: np.ndarray, sample_n: int = 2000) -> float:
    """Average cosine distance between random pairs. Higher = more discriminative."""
    rng = np.random.default_rng(42)
    idx = rng.choice(len(emb), min(sample_n, len(emb)), replace=False)
    sims = cosine_similarity(emb[idx])
    np.fill_diagonal(sims, 0)
    mask = sims != 0
    return float(1 - sims[mask].mean()) if mask.any() else 0.0


quality_results = {}
print(f"{'Strategy':<25} {'Isotropy':>10} {'AvgDist':>10}")
print("-" * 48)

for strategy in STRATEGY_NAMES:
    emb = raw_embeddings[strategy]
    iso = embedding_isotropy(emb)
    apd = avg_pairwise_distance(emb)
    quality_results[strategy] = {"isotropy": iso, "avg_pairwise_dist": apd}
    print(f"{strategy:<25} {iso:>10.4f} {apd:>10.4f}")

with open(RESULTS_DIR / "embedding_quality.json", "w") as f:
    json.dump(quality_results, f, indent=2)
print(f"\nQuality saved to {RESULTS_DIR / 'embedding_quality.json'}")

In [ ]:
# ─── Cell 12: Apply PCA & Save Embedding CSVs ────────────────────────────────
# Fit separate PCA per strategy (replicates repo's SASRecWithEmbeddings behavior).

embedding_csv_paths = {}  # strategy -> Path

for strategy in STRATEGY_NAMES:
    csv_path = EMB_DIR / f"beauty_{strategy}_pca{EMB_DIM}.csv"
    embedding_csv_paths[strategy] = csv_path

    if csv_path.exists():
        print(f"[SKIP] {strategy} PCA CSV already exists.")
        continue

    print(f"[PCA] {strategy}: {raw_embeddings[strategy].shape} -> {EMB_DIM}-dim...")
    t0 = time.time()
    pca = PCA(n_components=EMB_DIM, random_state=42)
    reduced = pca.fit_transform(raw_embeddings[strategy]).astype(np.float32)
    print(f"  Variance explained: {pca.explained_variance_ratio_.sum():.3f} | Time: {time.time()-t0:.1f}s")

    # Build CSV: ItemId (int) | embedding (JSON list)
    rows = [
        {
            "ItemId": item_int,
            "embedding": json.dumps([round(float(x), 6) for x in reduced[i]]),
        }
        for i, item_int in enumerate(all_item_ints)
    ]
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path}")

print("\nAll PCA embedding CSVs ready.")

In [ ]:
# ─── Cell 13: Run SASRec Experiments ─────────────────────────────────────────
# For each strategy: train LLM2SASRec, evaluate on test set, record metrics.
# Results saved incrementally in case of GPU timeout.

RESULTS_PATH = RESULTS_DIR / "promptcraft_beauty_results.json"

if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    print(f"Loaded existing results for {list(all_results.keys())}")
else:
    all_results = {}

optimizer_kwargs = {"learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY}
common_cfg = dict(
    N=MAX_SEQ_LEN,
    L=1,
    h=2,
    emb_dim=EMB_DIM,
    hidden_dim=EMB_DIM,
    drop_rate=DROP_RATE,
    activation="relu",
    num_epochs=NUM_EPOCHS,
    fit_batch_size=FIT_BATCH_SIZE,
    pred_batch_size=PRED_BATCH_SIZE,
    train_val_fraction=TRAIN_VAL_FRACTION,
    early_stopping_patience=EARLY_STOPPING,
    optimizer_kwargs=optimizer_kwargs,
    transformer_layer_kwargs={"layout": "NFDR"},
    is_verbose=VERBOSE,
    cores=CORES,
)

# ── Vanilla SASRec baseline (no LLM embeddings) ─────────────────────────────
if "vanilla_sasrec" not in all_results:
    print(f"\n{'='*60}")
    print("[RUN] Baseline: Vanilla SASRec (no LLM embeddings)")
    print(f"{'='*60}")
    try:
        t0 = time.time()
        model = SASRec(**common_cfg)
        model.train(train_data)
        recommendations = model.predict(test_prompts, top_k=max(TOP_KS))
        metrics = evaluate(recommendations, test_gts, top_ks=TOP_KS)
        elapsed = time.time() - t0
        all_results["vanilla_sasrec"] = {"metrics": metrics, "elapsed_seconds": round(elapsed, 1)}
        print(f"  Time: {elapsed/60:.1f} min")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")
        with open(RESULTS_PATH, "w") as f:
            json.dump(all_results, f, indent=2)
        del model
        gc.collect()
        tf.keras.backend.clear_session()
    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {e}")
        all_results["vanilla_sasrec"] = {"error": str(e)}
        with open(RESULTS_PATH, "w") as f:
            json.dump(all_results, f, indent=2)

for strategy in STRATEGY_NAMES:
    if strategy in all_results:
        print(f"[SKIP] {strategy} already in results.")
        continue

    print(f"\n{'='*60}")
    print(f"[RUN] Strategy: {strategy}")
    print(f"{'='*60}")

    try:
        t0 = time.time()
        model = LLM2SASRec(
            embedding_csv_path=str(embedding_csv_paths[strategy]),
            **common_cfg,
        )
        model.train(train_data)
        recommendations = model.predict(test_prompts, top_k=max(TOP_KS))
        metrics = evaluate(recommendations, test_gts, top_ks=TOP_KS)
        elapsed = time.time() - t0

        all_results[strategy] = {
            "metrics": metrics,
            "elapsed_seconds": round(elapsed, 1),
            "embedding_csv": str(embedding_csv_paths[strategy]),
        }

        print(f"  Time: {elapsed/60:.1f} min")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")

        # Save incrementally
        with open(RESULTS_PATH, "w") as f:
            json.dump(all_results, f, indent=2)

        # Free memory before next run
        del model
        gc.collect()
        tf.keras.backend.clear_session()

    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {e}")
        all_results[strategy] = {"error": str(e)}
        with open(RESULTS_PATH, "w") as f:
            json.dump(all_results, f, indent=2)

print(f"\nAll experiments done. Results saved to {RESULTS_PATH}")

In [ ]:
# ─── Cell 14: Results Summary Table ──────────────────────────────────────────

with open(RESULTS_PATH) as f:
    all_results = json.load(f)

baseline_ndcg10 = all_results.get("type1_title_only", {}).get("metrics", {}).get("NDCG@10", None)

# Show vanilla SASRec baseline separately
vanilla = all_results.get("vanilla_sasrec", {})
if "metrics" in vanilla:
    vm = vanilla["metrics"]
    print(f"\nVanilla SASRec (no LLM): NDCG@10={vm.get('NDCG@10',0):.4f}  HR@10={vm.get('HR@10',0):.4f}  MRR={vm.get('MRR',0):.4f}")
    if baseline_ndcg10:
        llm_gain = (baseline_ndcg10 - vm.get("NDCG@10", 0)) / max(vm.get("NDCG@10", 1e-9), 1e-9) * 100
        print(f"  LLM title-only vs vanilla: {llm_gain:+.1f}% NDCG@10")

print(f"\n{'='*75}")
print(f"PromptCraft-SeqRec Results — Amazon Beauty Dataset")
print(f"{'='*75}")
print(f"{'Strategy':<26} {'NDCG@10':>9} {'NDCG@20':>9} {'HR@10':>8} {'HR@20':>8} {'MRR':>8} {'vs Baseline':>12}")
print("-" * 75)

summary_rows = []
for strategy in STRATEGY_NAMES:
    result = all_results.get(strategy, {})
    if "error" in result:
        print(f"{strategy:<26} ERROR: {result['error'][:40]}")
        continue
    if "metrics" not in result:
        print(f"{strategy:<26} MISSING")
        continue
    m = result["metrics"]
    ndcg10 = m.get("NDCG@10", 0)
    ndcg20 = m.get("NDCG@20", 0)
    hr10   = m.get("HR@10", 0)
    hr20   = m.get("HR@20", 0)
    mrr_   = m.get("MRR", 0)
    if baseline_ndcg10 and strategy != "type1_title_only":
        delta = (ndcg10 - baseline_ndcg10) / baseline_ndcg10 * 100
        tag = f"{delta:+.1f}%"
    elif strategy == "type1_title_only":
        tag = "[BASELINE]"
    else:
        tag = "N/A"
    label = STRATEGY_LABELS.get(strategy, strategy)
    print(f"{label:<26} {ndcg10:>9.4f} {ndcg20:>9.4f} {hr10:>8.4f} {hr20:>8.4f} {mrr_:>8.4f} {tag:>12}")
    summary_rows.append({"Strategy": strategy, "Label": label,
                         "NDCG@10": ndcg10, "NDCG@20": ndcg20,
                         "HR@10": hr10, "HR@20": hr20, "MRR": mrr_})

if summary_rows:
    best = max(summary_rows, key=lambda r: r["NDCG@10"])
    print(f"\nBest strategy: {best['Strategy']} (NDCG@10={best['NDCG@10']:.4f})")
    if baseline_ndcg10:
        delta = (best["NDCG@10"] - baseline_ndcg10) / baseline_ndcg10 * 100
        print(f"Improvement over title-only baseline: {delta:+.1f}%")

In [ ]:
# ─── Cell 15: Visualization ───────────────────────────────────────────────────

if not summary_rows:
    print("No results to visualize.")
else:
    df_results = pd.DataFrame(summary_rows).set_index("Strategy")

    # ── Bar chart: NDCG@10, HR@10, MRR across strategies ────────────────────
    fig, ax = plt.subplots(figsize=(14, 5))
    metrics_to_plot = ["NDCG@10", "HR@10", "MRR"]
    colors = ["#2196F3", "#FF9800", "#4CAF50"]
    x = np.arange(len(df_results))
    width = 0.25

    for k, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
        vals = df_results[metric].values
        bars = ax.bar(x + k * width, vals, width, label=metric, color=color, alpha=0.85)
        baseline_val = df_results.loc["type1_title_only", metric] if "type1_title_only" in df_results.index else 0
        for i, (bar, val) in enumerate(zip(bars, vals)):
            label_str = STRATEGY_NAMES[i]
            if label_str == "type1_title_only":
                bar.set_edgecolor("red")
                bar.set_linewidth(2)
            if val == vals.max() and val > baseline_val:
                bar.set_edgecolor("darkgreen")
                bar.set_linewidth(2)
                delta = (val - baseline_val) / max(baseline_val, 1e-9) * 100
                ax.annotate(f"+{delta:.1f}%",
                           xy=(bar.get_x() + bar.get_width() / 2, val),
                           xytext=(0, 3), textcoords="offset points",
                           ha="center", fontsize=7, color="darkgreen", fontweight="bold")

    labels = [STRATEGY_LABELS.get(s, s) for s in df_results.index]
    ax.set_xticks(x + width)
    ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
    ax.set_ylabel("Score")
    ax.set_title("PromptCraft-SeqRec — Amazon Beauty\n(red border = baseline, green border + % = best improvement)")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    bar_path = FIGURES_DIR / "bar_beauty_metrics.pdf"
    plt.savefig(bar_path, dpi=300, bbox_inches="tight")
    plt.savefig(str(bar_path).replace(".pdf", ".png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Bar chart saved -> {bar_path}")

    # ── Heatmap: strategies vs metrics ───────────────────────────────────────
    metric_cols = ["NDCG@10", "NDCG@20", "HR@10", "HR@20", "MRR"]
    mat = df_results[metric_cols].values.astype(float)
    baseline_row = df_results.loc["type1_title_only", metric_cols].values if "type1_title_only" in df_results.index else np.zeros(len(metric_cols))
    delta_mat = (mat - baseline_row) / (np.abs(baseline_row) + 1e-9) * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    im1 = ax1.imshow(mat, cmap="YlOrRd", aspect="auto")
    ax1.set_xticks(range(len(metric_cols)))
    ax1.set_xticklabels(metric_cols)
    ax1.set_yticks(range(len(df_results)))
    ax1.set_yticklabels([STRATEGY_LABELS.get(s, s) for s in df_results.index])
    ax1.set_title("Absolute Scores")
    plt.colorbar(im1, ax=ax1)
    for i in range(len(df_results)):
        for j in range(len(metric_cols)):
            ax1.text(j, i, f"{mat[i,j]:.3f}", ha="center", va="center", fontsize=7)

    vcenter = 0
    vmin, vmax = float(delta_mat.min()), float(delta_mat.max())
    if vmin >= 0:
        vmin = min(-0.1, vmax - 0.2)
    if vmax <= 0:
        vmax = max(0.1, vmin + 0.2)
    if vmin == vmax:
        vmin, vmax = vmin - 1.0, vmax + 1.0
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    im2 = ax2.imshow(delta_mat, cmap="RdYlGn", aspect="auto", norm=norm)
    ax2.set_xticks(range(len(metric_cols)))
    ax2.set_xticklabels(metric_cols)
    ax2.set_yticks(range(len(df_results)))
    ax2.set_yticklabels([STRATEGY_LABELS.get(s, s) for s in df_results.index])
    ax2.set_title("% Change vs T1 Baseline")
    plt.colorbar(im2, ax=ax2)
    for i in range(len(df_results)):
        for j in range(len(metric_cols)):
            col = "white" if abs(delta_mat[i, j]) > 10 else "black"
            ax2.text(j, i, f"{delta_mat[i,j]:+.1f}%", ha="center", va="center", fontsize=7, color=col)

    plt.suptitle("PromptCraft-SeqRec: Prompt Strategy vs Metric Heatmap", fontsize=12, y=1.02)
    plt.tight_layout()
    heatmap_path = FIGURES_DIR / "heatmap_beauty.pdf"
    plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
    plt.savefig(str(heatmap_path).replace(".pdf", ".png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Heatmap saved -> {heatmap_path}")

    # ── Embedding quality scatter ─────────────────────────────────────────────
    if quality_results:
        fig, ax = plt.subplots(figsize=(8, 5))
        for strategy in STRATEGY_NAMES:
            if strategy not in quality_results or strategy not in df_results.index:
                continue
            x_val = quality_results[strategy]["avg_pairwise_dist"]
            y_val = df_results.loc[strategy, "NDCG@10"]
            label = STRATEGY_LABELS.get(strategy, strategy)
            color = "red" if strategy == "type1_title_only" else "steelblue"
            ax.scatter(x_val, y_val, s=100, color=color, zorder=5)
            ax.annotate(label, (x_val, y_val), textcoords="offset points",
                       xytext=(5, 5), fontsize=7)
        ax.set_xlabel("Avg Pairwise Distance (higher = more discriminative)")
        ax.set_ylabel("NDCG@10")
        ax.set_title("Embedding Discriminability vs Downstream Accuracy")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        scatter_path = FIGURES_DIR / "scatter_quality_vs_ndcg.pdf"
        plt.savefig(scatter_path, dpi=300, bbox_inches="tight")
        plt.savefig(str(scatter_path).replace(".pdf", ".png"), dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Scatter plot saved -> {scatter_path}")

print(f"\nAll figures in {FIGURES_DIR}/")

In [ ]:
# ─── Cell 16: Export Summary CSV ─────────────────────────────────────────────

if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    if baseline_ndcg10:
        df_summary["NDCG@10_delta_pct"] = (
            (df_summary["NDCG@10"] - baseline_ndcg10) / baseline_ndcg10 * 100
        ).round(2)
    csv_path = RESULTS_DIR / "promptcraft_beauty_summary.csv"
    df_summary.to_csv(csv_path, index=False)
    print(f"Summary CSV saved -> {csv_path}")
    display(df_summary)

print(f"""
Output files:
  Embeddings (raw):     {EMB_DIR}/beauty_*_raw.npy
  Embeddings (PCA-256): {EMB_DIR}/beauty_*_pca256.csv
  Quality metrics:      {RESULTS_DIR}/embedding_quality.json
  Full results:         {RESULTS_DIR}/promptcraft_beauty_results.json
  Summary CSV:          {RESULTS_DIR}/promptcraft_beauty_summary.csv
  Bar chart:            {FIGURES_DIR}/bar_beauty_metrics.png
  Heatmap:              {FIGURES_DIR}/heatmap_beauty.png
  Scatter:              {FIGURES_DIR}/scatter_quality_vs_ndcg.png
""")